# Brooklyn ambulance siting by quarter, with a Bertsimas–Sim demand budget

For each calendar quarter of 2025 we choose five new standby posts from the 20 candidates (pins 24–43). The 66 existing Brooklyn firehouses stay open. Each quarter gets its own model and can pick different posts.

**Demand.** Each of the 37 ZIP areas has 5 random demand points on its residential streets (`data/demand_points_185.csv`). The ZIP's calls are split evenly between its points. For quarter $q$, ZIP $j$ has weekly mean $\mu_{jq}$ and standard deviation $\sigma_{jq}$ (`data/quarterly_demand_by_zip.csv`; weekdays 9am–9pm, Monday–Friday weeks).

**Travel time.** Each demand point is served by its fastest open post (existing or new). There's no backup term and no 500 m position ball. $W_j$ is the average drive time over ZIP $j$'s points.

**Uncertainty (Bertsimas–Sim budget).** Weekly demand is $d_j = \mu_{jq} + k\,\sigma_{jq}\,z_j$ with $0 \le z_j \le 1$ and $\sum_j z_j \le \Gamma$. In words, each ZIP can be up to $k$ standard deviations busier than usual, but at most $\Gamma$ ZIPs at once. Only upward swings matter: drive times are positive, so a quieter ZIP never makes the cost worse, and the symmetric set $|z_j| \le 1$ gives the same model.

**Model for quarter $q$.** Minimize worst-case weekly call-minutes $\max_{z} \sum_j d_j W_j$. LP duality turns the inner maximum into
$$\min_{y,x,\lambda,\nu}\ \sum_j \mu_{jq} W_j + \Gamma\lambda + \sum_j \nu_j \quad \text{s.t.}\quad \lambda + \nu_j \ge k\,\sigma_{jq} W_j,\ \ \lambda,\nu_j \ge 0,$$
plus the usual assignment constraints. $\Gamma = 0$ is the plain (nominal) model, and $\Gamma = 37$ puts every ZIP at $\mu + k\sigma$.

**Benchmarks and checks.** For each quarter we compare status quo, busiest-first, greedy (same worst-case score), the nominal model and the robust model. Each is scored on the worst case and on the real weeks of that quarter. The real-week test is in-sample, because the means and standard deviations come from the same 2025 weeks.

Run from the project root with a Julia kernel. Put `demand_points_185.csv`, `quarterly_demand_by_zip.csv` and `weekly_calls_by_zip.csv` in `data/`. The first run routes 185 points from 86 sites (about 170 requests to `MATRIX_URL`). Later runs reuse the cache in `results/`.

In [1]:
# using Pkg
# Pkg.add("JSON3")

In [3]:
using CSV, DataFrames, JuMP, Gurobi, Printf
using Downloads, JSON3, SHA

## 1. Configuration

In [4]:
DATA_DIR   = "data"
OUTPUT_DIR = "results"

DEMAND_FILE    = joinpath(DATA_DIR, "demand_points_185.csv")        # point_id, area_id (ZIP), lat, lon
QUARTER_FILE   = joinpath(DATA_DIR, "quarterly_demand_v1.csv")  # area_id, quarter, mean/std of weekly calls
WEEKLY_FILE    = joinpath(DATA_DIR, "weekly_calls_by_zip.csv")      # area_id, quarter, week_start, calls_per_week
CANDIDATE_FILE = joinpath(DATA_DIR, "candidate_sites_v2.csv")       # site_id, lat, lon (pins 24-43)
EXISTING_FILE  = joinpath(DATA_DIR, "FDNY_Firehouse_Listing_20261115.csv")
ZIP_FILE       = joinpath(DATA_DIR, "modzcta.geojson")
BOROUGH        = "Brooklyn"

P           = 5                       # new standby posts per quarter
QUARTERS    = ["Q1", "Q2", "Q3", "Q4"]
K_STD       = 1.0                     # a ZIP can be up to K_STD standard deviations busier than its mean
GAMMA       = 9.0                     # main budget: at most GAMMA ZIPs at their high value at once
GAMMA_SWEEP = [0, 2, 4, 6, 9, 13, 20, 37]

TIME_LIMIT_SEC = 300
MATRIX_URL = "https://valhalla1.openstreetmap.de/sources_to_targets"
MAX_SNAP_M = 250
@assert K_STD >= 0 && GAMMA >= 0

## 2. Load demand points, quarterly demand and sites

In [5]:
avg(v) = sum(v) / length(v)

function in_ring(lon, lat, ring)
    inside = false
    for k in 2:length(ring)
        a, b = ring[k-1][1:2]
        c, d = ring[k][1:2]
        if (b > lat) != (d > lat) && lon < a + (lat - b) * (c - a) / (d - b)
            inside = !inside
        end
    end
    return inside
end

function in_zip(lon, lat, geom)
    polygons = geom.type == "Polygon" ? [geom.coordinates] : geom.coordinates
    return any(poly -> in_ring(lon, lat, first(poly)) &&
                       !any(hole -> in_ring(lon, lat, hole), Iterators.drop(poly, 1)), polygons)
end

isfile(DEMAND_FILE) || error("Missing $DEMAND_FILE. Run from the project root.")
demand = CSV.read(DEMAND_FILE, DataFrame; types=Dict(:point_id => String, :area_id => String))
all(c -> c in propertynames(demand), [:point_id, :area_id, :lat, :lon]) ||
    error("Demand points must have point_id, area_id, lat and lon columns.")
allunique(demand.point_id) || error("Demand point IDs must be unique.")
all(z -> occursin(r"^[0-9]{5}$", z), demand.area_id) || error("area_id must be a 5-digit ZIP.")

geo = JSON3.read(read(ZIP_FILE, String))
boundaries = Dict(String(f.properties.modzcta) => f.geometry for f in geo.features)
outside = [r.point_id for r in eachrow(demand)
           if !haskey(boundaries, r.area_id) || !in_zip(r.lon, r.lat, boundaries[r.area_id])]
isempty(outside) || error("Demand points outside their ZIP: $outside")
println("Demand points: ", nrow(demand), " | ZIPs: ", length(unique(demand.area_id)))
first(demand, 5)

Demand points: 185 | ZIPs: 37


Row,point_id,area_id,lat,lon
,String,String,Float64,Float64
1,11201-1,11201,40.6876,-73.9918
2,11201-2,11201,40.6948,-73.9915
3,11201-3,11201,40.6916,-73.9877
4,11201-4,11201,40.6973,-73.9925
5,11201-5,11201,40.6892,-73.9888


In [6]:
candidates = CSV.read(CANDIDATE_FILE, DataFrame)
candidates.site_id = string.(candidates.site_id)
J_sites = candidates.site_id

firehouses = CSV.read(EXISTING_FILE, DataFrame; normalizenames=true)
firehouses = dropmissing(filter(r -> r.Borough == BOROUGH, firehouses), [:Latitude, :Longitude])
existing = DataFrame(
    site_id = [@sprintf("FH_%03d", k) for k in 1:nrow(firehouses)],
    name    = firehouses.FacilityName,
    address = firehouses.FacilityAddress,
    lat     = Float64.(firehouses.Latitude),
    lon     = Float64.(firehouses.Longitude),
)
K_sites = existing.site_id
isempty(intersect(J_sites, K_sites)) || error("Candidate and existing site_ids overlap")

site_ids = vcat(K_sites, J_sites)          # existing first, then candidates
n_existing = length(K_sites)
sites = vcat(select(existing, :site_id, :lat, :lon), select(candidates, :site_id, :lat, :lon))
println("Candidate sites |J| = $(length(J_sites)),  existing firehouses |K| = $(n_existing)")

Candidate sites |J| = 20,  existing firehouses |K| = 66


## 3. Route every site to every demand point (cached)

In [7]:
matrix_location(point) = (lat=point.lat, lon=point.lon, search_cutoff=MAX_SNAP_M)
snap_meters(lat, lon, snapped_lat, snapped_lon) = 111195 * hypot(lat - snapped_lat, (lon - snapped_lon) * cosd(lat))

function route_points(sites, positions)
    result = fill(NaN, nrow(sites), length(positions))
    site_points = [(id=s.site_id, lat=s.lat, lon=s.lon) for s in eachrow(sites)]
    cache_key = bytes2hex(sha256(JSON3.write((site_points, positions, MATRIX_URL, MAX_SNAP_M))))
    mkpath(OUTPUT_DIR)
    path = joinpath(OUTPUT_DIR, "quarterly_points_$(cache_key[1:16]).csv")
    if isfile(path)
        cached = Matrix{Float64}(CSV.read(path, DataFrame))
        size(cached) == size(result) || error("Route cache dimensions do not match")
        return cached
    end
    println("Routing ", length(positions), " demand points from ", nrow(sites), " sites.")
    for site_batch in Iterators.partition(1:nrow(sites), 2)
        for target_batch in Iterators.partition(eachindex(positions), 48)
            payload = JSON3.write((
                sources=[matrix_location(sites[j, :]) for j in site_batch],
                targets=[matrix_location(positions[k]) for k in target_batch],
                costing="auto",
                costing_options=(auto=(shortest=true, disable_hierarchy_pruning=true),),
                units="kilometers", verbose=true,
            ))
            response_body = IOBuffer()
            response = Downloads.request(MATRIX_URL; method="POST",
                input=IOBuffer(payload), output=response_body, timeout=60,
                headers=["Content-Type" => "application/json",
                         "User-Agent" => "15C57-Brooklyn-EMS-Study/1.0",
                         "X-Client-Id" => "15C57-Brooklyn-EMS-Study"])
            body = String(take!(response_body))
            response.status == 200 || error("Valhalla HTTP $(response.status): $body")
            data = JSON3.read(body)
            data.units == "kilometers" || error("Expected Valhalla distances in kilometers")
            rows = data.sources_to_targets
            length(rows) == length(site_batch) || error("Incomplete Valhalla source batch")
            for (j, routes) in zip(site_batch, rows)
                length(routes) == length(target_batch) || error("Incomplete Valhalla target batch")
                for (k, route) in zip(target_batch, routes)
                    distance = get(route, :distance, nothing)
                    cost = get(route, :cost, nothing)
                    (distance === nothing || !isfinite(distance) || distance < 0) && continue
                    cost !== nothing && isfinite(cost) && abs(cost - 1000 * distance) <= 1.1 || continue
                    site, point = sites[j, :], positions[k]
                    any(key -> get(route, key, nothing) === nothing,
                        (:begin_lat, :begin_lon, :end_lat, :end_lon)) && continue
                    snap_meters(site.lat, site.lon, route.begin_lat, route.begin_lon) <= MAX_SNAP_M + 1 || continue
                    snap_meters(point.lat, point.lon, route.end_lat, route.end_lon) <= MAX_SNAP_M + 1 || continue
                    result[j, k] = 4 * distance / 1.609344      # minutes at 15 mph
                end
            end
        end
        (last(site_batch) % 10 == 0 || last(site_batch) == nrow(sites)) &&
            println("Routed ", last(site_batch), " / ", nrow(sites), " sites.")
    end
    CSV.write(path, DataFrame(result, :auto))
    return result
end

all_zips = sort(unique(demand.area_id))
positions = [(position_id=r.point_id, lat=Float64(r.lat), lon=Float64(r.lon)) for r in eachrow(demand)]
route_times = route_points(sites, positions)

# A point needs a route from every site; otherwise it's dropped and its ZIP's calls go to its other points.
routed = [all(isfinite, view(route_times, :, k)) for k in axes(route_times, 2)]
dropped = demand.point_id[.!routed]
isempty(dropped) || println("WARNING: dropped points with incomplete routes: ", join(dropped, ", "))
demand = demand[routed, :]
times = route_times[:, routed]                  # sites x points, minutes

zips = sort(unique(demand.area_id))
zip_index = Dict(z => j for (j, z) in enumerate(zips))
points_in_zip = [findall(==(z), demand.area_id) for z in zips]
zips == all_zips || error("These ZIPs have no routed points left: $(setdiff(all_zips, zips))")
println("Points used: ", size(times, 2), " | points per ZIP: ", sort(unique(length.(points_in_zip))))

Routing 185 demand points from 86 sites.
Routed 10 / 86 sites.
Routed 20 / 86 sites.
Routed 30 / 86 sites.
Routed 40 / 86 sites.
Routed 50 / 86 sites.
Routed 60 / 86 sites.
Routed 70 / 86 sites.
Routed 80 / 86 sites.
Routed 86 / 86 sites.
Points used: 185 | points per ZIP: [5]


In [8]:
# Weekly mean and standard deviation per ZIP and quarter.
qstats = CSV.read(QUARTER_FILE, DataFrame; types=Dict(:area_id => String, :quarter => String))
mu = fill(NaN, length(zips), length(QUARTERS))
sd = fill(NaN, length(zips), length(QUARTERS))
for r in eachrow(qstats)
    j = get(zip_index, r.area_id, 0); q = findfirst(==(r.quarter), QUARTERS)
    (j == 0 || q === nothing) && continue
    mu[j, q] = r.mean_weekly_calls
    sd[j, q] = r.std_weekly_calls
end
all(isfinite, mu) && all(isfinite, sd) || error("Missing mean or std for some ZIP and quarter")

# Real weekly calls, for the real-week test: one matrix (weeks x ZIPs) per quarter.
weekly = CSV.read(WEEKLY_FILE, DataFrame; types=Dict(:area_id => String, :quarter => String, :week_start => String))
week_calls = Dict{String, Matrix{Float64}}()
for quarter in QUARTERS
    sub = filter(r -> r.quarter == quarter && haskey(zip_index, r.area_id), weekly)
    weeks = sort(unique(sub.week_start))
    M = zeros(length(weeks), length(zips))
    for r in eachrow(sub)
        M[findfirst(==(r.week_start), weeks), zip_index[r.area_id]] = r.calls_per_week
    end
    week_calls[quarter] = M
end
DataFrame(quarter=QUARTERS,
          weekly_calls=[round(sum(mu[:, q]); digits=1) for q in eachindex(QUARTERS)],
          sum_of_zip_std=[round(sum(sd[:, q]); digits=1) for q in eachindex(QUARTERS)],
          weeks=[size(week_calls[qq], 1) for qq in QUARTERS])

Row,quarter,weekly_calls,sum_of_zip_std,weeks
,String,Float64,Float64,Int64
1,Q1,3847.7,451.0,14
2,Q2,3932.8,466.2,14
3,Q3,3910.0,473.0,14
4,Q4,3825.9,466.0,14


## 4. Model, scoring and benchmarks

`robust_score` evaluates any set of new posts the same way the model does: it finds the adversary's worst demand by putting the $\Gamma$ largest values of $k\sigma_j W_j$ at their maximum (and a fraction of the next one). The model's objective must match it exactly.

In [9]:
# Average drive time to each ZIP's points when the existing firehouses and `selected` are open.
function zip_times(selected)
    open_sites = vcat(collect(1:n_existing), selected)
    point_time = vec(minimum(view(times, open_sites, :); dims=1))
    return [avg(point_time[points_in_zip[j]]) for j in eachindex(zips)]
end

# The adversary's choice: z_j = 1 for the Γ ZIPs where extra calls cost most, a fraction for the next.
function worst_case_z(v, Γ)
    z = zeros(length(v))
    left = Float64(Γ)
    for j in sortperm(v; rev=true)
        left <= 0 && break
        z[j] = min(1.0, left)
        left -= z[j]
    end
    return z
end

function robust_score(selected, q, Γ)
    W = zip_times(selected)
    z = worst_case_z(K_STD .* sd[:, q] .* W, Γ)
    d = mu[:, q] .+ K_STD .* sd[:, q] .* z          # worst-case weekly calls per ZIP
    total = sum(d .* W)
    return (worst=total,                            # worst-case call-minutes per week
            per_call=total / sum(d),                # minutes per call in that worst-case week
            nominal=sum(mu[:, q] .* W) / sum(mu[:, q]),
            W=W, z=z)
end

function solve_quarter(q, Γ)
    n_sites, n_points = size(times)
    model = Model(Gurobi.Optimizer)
    set_silent(model)
    set_time_limit_sec(model, TIME_LIMIT_SEC)
    set_attribute(model, "MIPGap", 0.0)   # rival layouts differ by fractions of a percent
    @variable(model, y[1:n_sites], Bin)
    @constraint(model, [j in 1:n_existing], y[j] == 1)
    @constraint(model, sum(y[j] for j in n_existing+1:n_sites) == P)

    point_time = Vector{AffExpr}(undef, n_points)
    for k in 1:n_points
        # Only the nearest firehouse and candidates faster than it can serve point k.
        nearest = argmin(view(times, 1:n_existing, k))
        eligible = vcat(nearest, [j for j in n_existing+1:n_sites if times[j, k] < times[nearest, k]])
        x = @variable(model, [eligible], lower_bound = 0)
        @constraint(model, sum(x[j] for j in eligible) == 1)
        @constraint(model, [j in eligible], x[j] <= y[j])
        point_time[k] = @expression(model, sum(times[j, k] * x[j] for j in eligible))
    end
    W = [@expression(model, sum(point_time[k] for k in points_in_zip[j]) / length(points_in_zip[j]))
         for j in eachindex(zips)]

    # Robust counterpart of max over the budget set (LP duality).
    @variable(model, λ >= 0)
    @variable(model, ν[eachindex(zips)] >= 0)
    @constraint(model, [j in eachindex(zips)], λ + ν[j] >= K_STD * sd[j, q] * W[j])
    @objective(model, Min, sum(mu[j, q] * W[j] for j in eachindex(zips)) + Γ * λ + sum(ν))

    optimize!(model)
    is_solved_and_feasible(model) || error("Solve failed: $(termination_status(model))")
    selected = [j for j in n_existing+1:n_sites if value(y[j]) > 0.5]
    return (selected=selected, objective=objective_value(model), gap=relative_gap(model))
end

function greedy_layout(q, Γ)
    selected = Int[]
    remaining = collect(n_existing+1:length(site_ids))
    for _ in 1:P
        scores = [robust_score(vcat(selected, j), q, Γ).worst for j in remaining]
        push!(selected, splice!(remaining, argmin(scores)))
    end
    return selected
end

# For each of the P busiest ZIPs this quarter, take the unused candidate with the lowest average time to its points.
function busiest_first(q)
    selected = Int[]
    remaining = collect(n_existing+1:length(site_ids))
    for j in sortperm(mu[:, q]; rev=true)[1:P]
        push!(selected, splice!(remaining, argmin([avg(times[s, points_in_zip[j]]) for s in remaining])))
    end
    return selected
end

# Minutes per call in each real week of the quarter.
function week_eval(selected, q)
    W = zip_times(selected)
    M = week_calls[QUARTERS[q]]
    per_call = (M * W) ./ vec(sum(M; dims=2))
    return (avg=avg(per_call), worst=maximum(per_call))
end

site_list(layout) = isempty(layout) ? "none" : join(sort(site_ids[layout]; by=s -> parse(Int, s)), ", ")

site_list (generic function with 1 method)

## 5. Solve each quarter at the main budget

In [10]:
comparison = DataFrame(quarter=String[], method=String[], sites=String[],
    worst_case_call_minutes=Float64[], worst_case_min_per_call=Float64[],
    nominal_min_per_call=Float64[], real_weeks_avg_min_per_call=Float64[], real_weeks_worst_min_per_call=Float64[])
selected_sites = DataFrame()

for (q, quarter) in enumerate(QUARTERS)
    sol = solve_quarter(q, GAMMA)
    @assert isapprox(robust_score(sol.selected, q, GAMMA).worst, sol.objective; rtol=1e-6)
    layouts = ["Status quo" => Int[],
               "Busiest-first" => busiest_first(q),
               "Greedy" => greedy_layout(q, GAMMA),
               "Nominal model (Γ = 0)" => solve_quarter(q, 0.0).selected,
               "Robust model" => sol.selected]
    for (method, layout) in layouts
        sc = robust_score(layout, q, GAMMA); wk = week_eval(layout, q)
        push!(comparison, (quarter, method, site_list(layout), round(sc.worst; digits=1),
              round(sc.per_call; digits=4), round(sc.nominal; digits=4), round(wk.avg; digits=4), round(wk.worst; digits=4)))
    end
    g = robust_score(layouts[3].second, q, GAMMA).worst
    @printf("%s: robust picks %s | greedy is %.3f%% worse on worst-case call-minutes\n",
            quarter, site_list(sol.selected), 100 * (g / sol.objective - 1))
    picked = candidates[sol.selected .- n_existing, :]
    insertcols!(picked, 1, :quarter => quarter)
    append!(selected_sites, picked)
end

mkpath(OUTPUT_DIR)
CSV.write(joinpath(OUTPUT_DIR, "quarterly_comparison.csv"), comparison)
CSV.write(joinpath(OUTPUT_DIR, "quarterly_selected_sites.csv"), selected_sites)
comparison

Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Q1: robust picks 30, 35, 39, 40, 42 | greedy is -0.000% worse on worst-case call-minutes
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Q2: robust picks 30, 35, 39, 40, 42 | greedy is -0.000% worse on worst-case call-minutes
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Q3: robust picks 30, 35, 37, 3

Row,quarter,method,sites,worst_case_call_minutes,worst_case_min_per_call,nominal_min_per_call,real_weeks_avg_min_per_call,real_weeks_worst_min_per_call
,String,String,String,Float64,Float64,Float64,Float64,Float64
1,Q1,Status quo,none,9706.8,2.431,2.4142,2.4178,2.4728
2,Q1,Busiest-first,"25, 26, 30, 36, 42",9384.9,2.3525,2.334,2.3372,2.3838
3,Q1,Greedy,"30, 35, 39, 40, 42",9266.8,2.3191,2.3062,2.3093,2.3537
4,Q1,Nominal model (Γ = 0),"30, 35, 39, 40, 42",9266.8,2.3191,2.3062,2.3093,2.3537
5,Q1,Robust model,"30, 35, 39, 40, 42",9266.8,2.3191,2.3062,2.3093,2.3537
6,Q2,Status quo,none,9885.8,2.4202,2.4078,2.408,2.4283
7,Q2,Busiest-first,"26, 29, 30, 36, 42",9573.3,2.3495,2.3295,2.3297,2.3533
8,Q2,Greedy,"30, 35, 39, 40, 42",9461.7,2.3208,2.3037,2.3044,2.327
9,Q2,Nominal model (Γ = 0),"30, 35, 39, 40, 42",9461.7,2.3208,2.3037,2.3044,2.327


## 6. How the choice changes with Γ

Γ = 0 is the nominal model, and Γ = 37 puts every ZIP at its high value. As a guide, Bertsimas and Sim show that if ZIP swings are independent and symmetric, the chance that the real cost exceeds the robust value is roughly $1-\Phi\big((\Gamma-1)/\sqrt{37}\big)$. That gives about 10% at Γ = 9 and about 5% at Γ = 11. Their strict bound $e^{-\Gamma^2/74}$ needs Γ ≈ 13 for 10%. Weekly calls can go beyond ±1σ, so treat these numbers only as a guide.

In [11]:
sweep = DataFrame(quarter=String[], gamma=Float64[], robust_sites=String[], robust_worst=Float64[],
                  greedy_sites=String[], greedy_worst=Float64[], greedy_gap_pct=Float64[])
for (q, quarter) in enumerate(QUARTERS), Γ in Float64.(GAMMA_SWEEP)
    sol = solve_quarter(q, Γ)
    g = greedy_layout(q, Γ)
    gw = robust_score(g, q, Γ).worst
    push!(sweep, (quarter, Γ, site_list(sol.selected), round(sol.objective; digits=1),
                  site_list(g), round(gw; digits=1), round(100 * (gw / sol.objective - 1); digits=3)))
end
CSV.write(joinpath(OUTPUT_DIR, "quarterly_gamma_sweep.csv"), sweep)
sweep

Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic license - for non-commercial use only - expires 2027-08-26
Set parameter Username
Set parameter LicenseID to value 2857454
Academic lic

Row,quarter,gamma,robust_sites,robust_worst,greedy_sites,greedy_worst,greedy_gap_pct
,String,Float64,String,Float64,String,Float64,Float64
1,Q1,0.0,"30, 35, 39, 40, 42",8873.6,"30, 35, 39, 40, 42",8873.6,-0.0
2,Q1,2.0,"30, 35, 39, 40, 42",8977.6,"30, 35, 39, 40, 42",8977.6,-0.0
3,Q1,4.0,"30, 35, 39, 40, 42",9074.1,"30, 35, 39, 40, 42",9074.1,0.0
4,Q1,6.0,"30, 35, 39, 40, 42",9152.1,"30, 35, 39, 40, 42",9152.1,-0.0
5,Q1,9.0,"30, 35, 39, 40, 42",9266.8,"30, 35, 39, 40, 42",9266.8,-0.0
6,Q1,13.0,"30, 35, 39, 40, 42",9402.1,"30, 35, 39, 40, 42",9402.1,0.0
7,Q1,20.0,"30, 35, 39, 40, 42",9605.7,"30, 35, 39, 40, 42",9605.7,-0.0
8,Q1,37.0,"30, 35, 39, 40, 42",9930.0,"30, 35, 39, 40, 42",9930.0,0.0
9,Q2,0.0,"30, 35, 39, 40, 42",9060.0,"30, 35, 39, 40, 42",9060.0,-0.0
